# 03 — Detecção de regimes via HMM

Especificação do pré-projeto (seção 3.3): HMM gaussiano (covariância cheia)
sobre observações multivariadas — **retorno agregado da carteira +
volatilidade realizada** —, 2-4 estados via BIC/AIC + inspeção econômica,
Baum-Welch com múltiplas inicializações, Viterbi (caracterização in-sample),
Forward (inferência prospectiva, sem lookahead), robustez via bootstrap em
blocos.

**Este é o fit único/exploratório da Semana 4** (in-sample, para
caracterizar regimes) — o refit causal walk-forward de verdade, que gera a
feature que efetivamente alimenta o LSTM sem vazar informação futura para
os parâmetros do modelo, é escopo da Semana 6.

Duas especificações de observação, ambas do retorno agregado equal-weighted
(já convertido para BRL — ver notebook 02):
- **Principal (3D)**: `return_1d`, `realized_vol_21d`, `realized_vol_63d`
- **Robustez/parcimoniosa (2D)**: `return_1d`, `realized_vol_21d`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.hmm import (
    block_bootstrap_stability,
    characterize_regimes,
    fit_gaussian_hmm,
    forward_filter,
    select_n_states,
    viterbi_decode,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

portfolio_full = pd.read_parquet(PROCESSED_DIR / "portfolio_return.parquet")
valid = portfolio_full.notna().all(axis=1)
groups = valid.ne(valid.shift()).cumsum()
lengths = valid[valid].groupby(groups[valid]).size().tolist()
portfolio = portfolio_full.loc[valid]
print("Sequ?ncias cont?guas:", lengths)
portfolio.tail()

Sequ?ncias cont?guas: [207, 1407, 2328]


,return_1d,realized_vol_21d,realized_vol_63d
2024-03-22,-0.013997,0.154832,0.138326
2024-03-25,0.002428,0.149277,0.137861
2024-03-26,-0.002924,0.142462,0.137405
2024-03-27,-0.002692,0.123120,0.137495
2024-03-28,0.021830,0.142522,0.144366


## Especificação principal (3D): retorno + vol. realizada 21d + 63d

In [2]:
cols_3d = ["return_1d", "realized_vol_21d", "realized_vol_63d"]
obs_3d = portfolio[cols_3d].values

selection_3d = select_n_states(obs_3d, state_range=(2, 3, 4), n_init=20, random_state=42, lengths=lengths)
selection_3d

Model is not converging.  Current: 29373.312674386154 is not greater than 29373.372455438992. Delta is -0.05978105283793411


Model is not converging.  Current: 29351.44605021518 is not greater than 29351.456701233623. Delta is -0.010651018441421911


Model is not converging.  Current: 29351.494218114443 is not greater than 29351.52714525134. Delta is -0.03292713689734228


Model is not converging.  Current: 27468.92827200354 is not greater than 27468.93305021553. Delta is -0.004778211990924319


Model is not converging.  Current: 29494.911413092468 is not greater than 29494.91269417625. Delta is -0.0012810837833967526


Model is not converging.  Current: 30331.171759368826 is not greater than 30331.171973665052. Delta is -0.00021429622574942186


,log_likelihood,bic,aic
n_states,,,
2,27414.526542,-54655.184771,-54787.053084
3,29355.735923,-58421.691324,-58641.471846
4,30912.833170,-61403.414722,-61723.666340


In [3]:
n_states_3d = selection_3d["bic"].idxmin()
print(f"BIC seleciona {n_states_3d} estados (AIC sugere {selection_3d['aic'].idxmin()})")

model_3d = fit_gaussian_hmm(obs_3d, n_states=n_states_3d, n_init=30, random_state=42, lengths=lengths)
regime_summary_3d = characterize_regimes(model_3d, obs_3d, cols_3d)
regime_summary_3d

Model is not converging.  Current: 29494.911413092468 is not greater than 29494.91269417625. Delta is -0.0012810837833967526


BIC seleciona 4 estados (AIC sugere 4)


Model is not converging.  Current: 30331.171759368313 is not greater than 30331.171973664517. Delta is -0.00021429620392154902


Model is not converging.  Current: 29438.37960577945 is not greater than 29438.38302360267. Delta is -0.00341782321993378


,persistence,expected_duration_days,mean_return_1d,mean_realized_vol_21d,mean_realized_vol_63d
state_rank,,,,,
0,0.989500,95.234781,0.000429,0.139431,0.141638
1,0.979184,48.041019,0.000627,0.172339,0.177315
2,0.981468,53.960690,0.000533,0.220678,0.224729
3,0.984309,63.731975,0.001871,0.401272,0.508475


## Viterbi (in-sample) vs. Forward (prospectivo, sem lookahead)

In [4]:
viterbi_states = viterbi_decode(model_3d, obs_3d, lengths=lengths)
forward_probs = forward_filter(model_3d, obs_3d, lengths=lengths)
forward_state = forward_probs.argmax(axis=1)

regimes = pd.DataFrame(
    {
        "viterbi_state": viterbi_states,
        "forward_state": forward_state,
        **{f"forward_prob_state_{s}": forward_probs[:, s] for s in range(n_states_3d)},
    },
    index=portfolio.index,
)

agreement = (regimes["viterbi_state"] == regimes["forward_state"]).mean()
print(f"Viterbi (in-sample) concorda com Forward (prospectivo) em {agreement:.1%} dos dias")
print("(divergências esperadas nas transições de regime — Viterbi vê o futuro próximo, Forward não)")

regimes.to_parquet(PROCESSED_DIR / "hmm_regimes_insample.parquet")
regimes.tail()

Viterbi (in-sample) concorda com Forward (prospectivo) em 96.8% dos dias
(divergências esperadas nas transições de regime — Viterbi vê o futuro próximo, Forward não)


,viterbi_state,forward_state,forward_prob_state_0,forward_prob_state_1,forward_prob_state_2,forward_prob_state_3
2024-03-22,0,0,0.999985,0.000015,6.862478e-07,3.676674e-16
2024-03-25,0,0,0.999994,0.000006,2.167608e-07,2.716713e-16
2024-03-26,0,0,0.999995,0.000005,1.837547e-07,8.174030e-17
2024-03-27,0,0,0.999996,0.000004,1.567553e-07,7.276132e-17
2024-03-28,0,0,0.999939,0.000059,1.375704e-06,4.624091e-16


## Robustez: bootstrap em blocos

Avalia estabilidade do nº de estados escolhido por BIC e da sequência de
regimes (Viterbi), com estados alinhados por volatilidade entre reamostragens
e comparados via % de concordância + Adjusted Rand Index.

In [5]:
stability_3d = block_bootstrap_stability(
    obs_3d, state_range=(2, 3, 4), block_length=21, n_bootstrap=50, n_init=5, random_state=42, lengths=lengths
)
stability_3d

Model is not converging.  Current: 29373.312674386154 is not greater than 29373.372455438992. Delta is -0.05978105283793411


Model is not converging.  Current: 29351.44605021518 is not greater than 29351.456701233623. Delta is -0.010651018441421911


Model is not converging.  Current: 29494.911413092468 is not greater than 29494.91269417625. Delta is -0.0012810837833967526


Model is not converging.  Current: 29494.911413092468 is not greater than 29494.91269417625. Delta is -0.0012810837833967526


Model is not converging.  Current: 25679.011037625096 is not greater than 25679.835122458753. Delta is -0.8240848336572526


...

Model is not converging.  Current: 29594.424061774724 is not greater than 29594.43218676492. Delta is -0.008124990195938153


Model is not converging.  Current: 29594.425106856776 is not greater than 29594.43166803231. Delta is -0.006561175534443464


Model is not converging.  Current: 31197.78686835281 is not greater than 31197.790054635818. Delta is -0.0031862830073805526


Model is not converging.  Current: 31197.786868352832 is not greater than 31197.7900546358. Delta is -0.0031862829673627857


{'reference_n_states': 4,
 'selected_n_states_distribution': 3    0.020408
 4    0.979592
 Name: proportion, dtype: float64,
 'regime_agreement_pct_mean': 0.6867178251310672,
 'regime_agreement_pct_std': 0.1297990717290063,
 'adjusted_rand_index_mean': 0.46800024740490653,
 'adjusted_rand_index_std': 0.12059802153948403,
 'n_bootstrap_completed': 49,
 'n_bootstrap_failed': 1,
 'n_bootstrap_same_n_states': 48}

## Especificação de robustez (2D): só retorno + vol. realizada 21d

Versão mais parcimoniosa — `realized_vol_21d` e `realized_vol_63d` são
fortemente correlacionadas (Semana 3), então o modelo 3D pode estar
efetivamente perto de um 2D com peso redobrado na vol. de curto prazo.
Comparar os regimes das duas especificações é um teste de robustez barato.

In [6]:
cols_2d = ["return_1d", "realized_vol_21d"]
obs_2d = portfolio[cols_2d].values

selection_2d = select_n_states(obs_2d, state_range=(2, 3, 4), n_init=20, random_state=42, lengths=lengths)
n_states_2d = selection_2d["bic"].idxmin()
model_2d = fit_gaussian_hmm(obs_2d, n_states=n_states_2d, n_init=30, random_state=42, lengths=lengths)

viterbi_2d = viterbi_decode(model_2d, obs_2d, lengths=lengths)
agreement_2d_vs_3d = (viterbi_2d == viterbi_states).mean() if n_states_2d == n_states_3d else None

print(f"2D seleciona {n_states_2d} estados (3D selecionou {n_states_3d})")
if agreement_2d_vs_3d is not None:
    print(f"Concordância Viterbi 2D vs. 3D: {agreement_2d_vs_3d:.1%}")
else:
    print("Nº de estados difere entre 2D e 3D — comparação direta de rótulo não é direta.")

characterize_regimes(model_2d, obs_2d, cols_2d)

Model is not converging.  Current: 19165.926382746184 is not greater than 19165.931578034437. Delta is -0.005195288253162289


Model is not converging.  Current: 20466.763324345706 is not greater than 20466.80508563437. Delta is -0.04176128866311046


Model is not converging.  Current: 21648.527707272893 is not greater than 21648.539430634424. Delta is -0.0117233615310397


Model is not converging.  Current: 21648.621332883027 is not greater than 21648.70942409042. Delta is -0.08809120739169884


Model is not converging.  Current: 21648.527707272893 is not greater than 21648.539430634424. Delta is -0.0117233615310397


Model is not converging.  Current: 21648.621332883027 is not greater than 21648.70942409042. Delta is -0.08809120739169884


Model is not converging.  Current: 20461.24294580962 is not greater than 20461.259431838058. Delta is -0.016486028438521316


Model is not converging.  Current: 21649.04637324042 is not greater than 21649.061617102656. Delta is -0.015243862235365668


2D seleciona 4 estados (3D selecionou 4)
Concordância Viterbi 2D vs. 3D: 55.1%


,persistence,expected_duration_days,mean_return_1d,mean_realized_vol_21d
state_rank,,,,
0,0.957137,23.330036,0.000338,0.124812
1,0.943708,17.764529,0.000565,0.165059
2,0.954097,21.784983,0.000612,0.215915
3,0.962911,26.962153,0.001373,0.347327


## Salvar relatórios

In [7]:
selection_3d.to_csv(TABLES_DIR / "hmm_state_selection_3d.csv")
regime_summary_3d.to_csv(TABLES_DIR / "hmm_regime_characterization_3d.csv")

pd.Series(stability_3d).drop("selected_n_states_distribution").to_csv(TABLES_DIR / "hmm_bootstrap_stability_3d.csv")
stability_3d["selected_n_states_distribution"].to_csv(TABLES_DIR / "hmm_bootstrap_n_states_distribution_3d.csv")

print("Relatórios salvos em", TABLES_DIR)
print("Regimes (Viterbi + Forward) salvos em", PROCESSED_DIR / "hmm_regimes_insample.parquet")

Relatórios salvos em C:\Python\hmm_lstm_TCC\reports\tables
Regimes (Viterbi + Forward) salvos em C:\Python\hmm_lstm_TCC\data\processed\hmm_regimes_insample.parquet
